In [ ]:
# pdf 문서에서 텍스트 또는 이미지 읽기
!pip install langchain langchain-community pymupdf pytesseract pillow transformers

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
import fitz
from PIL import Image
import pytesseract
import os
import io
import matplotlib.pyplot as plt


In [ ]:
# 텍스트 읽기 1) PyMuPDFLoader
pdf_path = 'aiethics.pdf'
loader = PyMuPDFLoader(pdf_path)
# Document : LangChain 의 기본 문서 객체입니다.
# 속성
# page_content: 문서의 내용을 나타내는 문열입니다.
# metadata: 문서의 메타데이터를 나타내는 딕셔너리입니다.

documents = loader.load()  # 각 페이지가 Document 객체로 변환
print(documents)   # [Document(metadata={'producer': 'Adobe PDF ...

# pdf의 텍스트만 출력
full_text = "\n\n".join([doc.page_content.strip() for doc in documents if doc.page_content.strip()])
print(full_text)

In [ ]:
# 텍스트 읽기 2) fitz
from langchain_core.documents import Document

doc = fitz.open('aiethics.pdf')
# print(doc)

documents = []
for i, page in enumerate(doc):
    text = page.get_text()
    metadata = {
        'source':pdf_path,
        'page': i + 1,
        'total_pages':len(doc),
        'author':doc.metadata.get('author', ''),
        'title':doc.metadata.get('title', ''),
        'producer':doc.metadata.get('producer', '')
    }

    if text.strip():
        documents.append(Document(page_content=text.strip(), metadata=metadata))

print(f'총 문서 수 : {len(documents)}')
print(documents[0])
full_text = "\n\n".join([doc.page_content.strip() for doc in documents if doc.page_content.strip()])
print(full_text)


In [ ]:
# 이미지 읽기
doc = fitz.open(pdf_path)
ocr_results = []  # ocr 결과 저장

for page_num, page in enumerate(doc):
    images = page.get_images(full=True)
    if not images:
        continue
    # print(images)

    for i, img in enumerate(images):
        xref = img[0]   # 이미지의 참조 id
        base_image = doc.extract_image(xref)
        image_bytes = base_image['image']  # 이미지의 raw byte data를 가져옴
        image = Image.open(io.BytesIO(image_bytes))

        # 문서에서 읽은 이미지를 각각 파일로 저장
        save_dirs = 'imgs'
        os.makedirs(save_dirs, exist_ok=True)

        image_filename = f'page_{page_num + 1}_image_{i + 1}.jpg'
        image_path = os.path.join(save_dirs, image_filename)
        image.save(image_path)
        #print(f'이미지 저장 완료:{image_path}')

        # OCR(광학문자인식)을 적용
        # 이미지를 분석 후 텍스트 + 위치 + confidence 등의 정보를 dict type으로 반환
        data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
        # print('data : ', data)
        # print(data['text'])

        # 신뢰도가 60 이상인 단어만 골라(잡음제거) 문자열로 결합
        extracted_text = ' '.join([text for j, text in enumerate(data['text']) if int(data['conf'][j]) >= 60])
        # print(extracted_text)

        ocr_results.append(
            {
                'page':page_num + 1,
                'image_index' :  i + 1,
                'text':extracted_text
            }
        )

        # 이미지 보기
        plt.figure(figsize=(3, 3))
        plt.imshow(image)
        for j in range(len(data['text'])):
            if int(data['conf'][j]) >= 60:   # ocr 텍스트 중 신뢰도 60 이상인 것만 시각적으로 표시
                (x, y, w, h) = (data['left'][j], data['top'][j],data['width'][j],data['height'][j])
                plt.gca().add_patch(plt.Rectangle((x, y), w, h,
                                    edgecolor='red', facecolor='none', linewidth=1))
        plt.axis('off')
        plt.title(f'ocr result on Page {page_num + 1}, Image{i + 1}')
        plt.show()

print(ocr_results[:2])